In [7]:
# 1. Clone the repository and change directory into it
!git clone https://github.com/Xerophayze/TTS-Story.git

Cloning into 'TTS-Story'...
remote: Enumerating objects: 2273, done.
remote: Counting objects: 100% (775/775), done.
remote: Compressing objects: 100% (303/303), done.
remote: Total 2273 (delta 535), reused 569 (delta 464), pack-reused 1498 (from 2)
Receiving objects: 100% (2273/2273), 27.71 MiB | 26.92 MiB/s, done.
Resolving deltas: 100% (1199/1199), done.


In [8]:
# 2. Install required system dependencies for audio processing
!apt-get update -qq && apt-get install -y espeak-ng sox libsox-dev ffmpeg

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
The following additional packages will be installed:
  espeak-ng-data libao-common libao4 libespeak-ng1 libid3tag0 libmad0
  libopencore-amrnb0 libopencore-amrwb0 libpcaudio0 libsonic0 libsox-fmt-all
  libsox-fmt-alsa libsox-fmt-ao libsox-fmt-base libsox-fmt-mp3 libsox-fmt-oss
  libsox-fmt-pulse libsox3 libwavpack1
Suggested packages:
  libaudio2 libsndio6.1
The following NEW packages will be installed:
  espeak-ng espeak-ng-data libao-common libao4 libespeak-ng1 libid3tag0
  libmad0 libopencore-amrnb0 libopencore-amrwb0 libpcaudio0 libsonic0
  libsox-dev libsox-fmt-all libsox-fmt-alsa libsox-fmt-ao libsox-fmt-base
  libsox-fmt-mp3 libsox-fmt-oss li

In [15]:
# 3. Explicitly install edge-tts and other dependencies to ensure successful import
!pip install --quiet edge-tts nest-asyncio
import os
if os.path.exists('TTS-Story/requirements.txt'):
    !pip install -r TTS-Story/requirements.txt
else:
    !pip install flask edge-tts gtts soundfile pydub

In [10]:
# 4. Install pyngrok to expose the local server publicly
!pip install pyngrok

In [16]:
# 5. Query available Bengali voices and update the config.json file
import json
import os
import asyncio
import edge_tts
import nest_asyncio
nest_asyncio.apply()

async def get_bengali_voices():
    voices = await edge_tts.VoicesManager.create()
    bn_voices = voices.find(Language="bn")
    return [v["Name"] for v in bn_voices]

try:
    bn_voices_list = asyncio.get_event_loop().run_until_complete(get_bengali_voices())
except Exception as e:
    bn_voices_list = []
    print("Error fetching voices:", e)

print("Found Bengali Voices:", bn_voices_list)

default_voice = bn_voices_list[0] if bn_voices_list else "en-US-AriaNeural"
print(f"Selected default voice: {default_voice}")

config_path = 'TTS-Story/config.json'
config_data = {}
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        try:
            config_data = json.load(f)
        except Exception:
            pass

config_data["tts_engine"] = "edge_tts"
config_data["edge_tts_default_voice"] = default_voice

with open(config_path, 'w') as f:
    json.dump(config_data, f, indent=4)
print("config.json updated successfully!")

Found Bengali Voices: ['Microsoft Server Speech Text to Speech Voice (bn-BD, NabanitaNeural)', 'Microsoft Server Speech Text to Speech Voice (bn-BD, PradeepNeural)', 'Microsoft Server Speech Text to Speech Voice (bn-IN, BashkarNeural)', 'Microsoft Server Speech Text to Speech Voice (bn-IN, TanishaaNeural)']
Selected default voice: Microsoft Server Speech Text to Speech Voice (bn-BD, NabanitaNeural)
config.json updated successfully!


In [12]:
# 6. Start the Flask application in the background and log output
import subprocess
import time

# Run the application in the background
with open("flask_app.log", "w") as log_file:
    process = subprocess.Popen(
        ["python", "app.py"],
        cwd="TTS-Story",
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

time.sleep(3) # Let the server start up
print("Flask app launched in background. Logs are being written to flask_app.log")

Flask app launched in background. Logs are being written to flask_app.log


In [ ]:
# 7. Open a public tunnel to Port 5000 using pyngrok
from pyngrok import ngrok
import getpass

# Securely prompt for ngrok token if not already authenticated
print("To use pyngrok, you need a free authtoken from https://dashboard.ngrok.com/get-started/your-authtoken")
authtoken = getpass.getpass("Enter your ngrok Authtoken: ")

if authtoken:
    !ngrok config add-authtoken {authtoken}

try:
    # Disconnect any existing tunnels
    ngrok.kill()
    public_url = ngrok.connect(5000)
    print("\n=======================================")
    print(f"Click this URL to open the TTS-Story web interface: {public_url.public_url}")
    print("=======================================\n")
except Exception as e:
    print("Error creating ngrok tunnel:", e)

To use pyngrok, you need a free authtoken from https://dashboard.ngrok.com/get-started/your-authtoken


In [14]:
# 8. Verify service health, show available Bengali voices, or output Flask server logs if offline
import requests
import os

print("--- Diagnostic Report ---")
print("Available Bengali Voices for reference:", bn_voices_list if 'bn_voices_list' in locals() else "No voices initialized")

try:
    # Try to access the local flask service health endpoint
    response = requests.get("http://127.0.0.1:5000/api/health", timeout=5)
    print("Health Check Status Code:", response.status_code)
    print("Response:", response.json())
except Exception as e:
    print("\nCould not reach health check endpoint. Checking Flask logs...")
    log_path = "flask_app.log"
    if os.path.exists(log_path):
        with open(log_path, "r") as f:
            logs = f.readlines()
            print("\nLast 20 lines of flask_app.log:")
            print("".join(logs[-20:]))
    else:
        print("No log file found at flask_app.log")

Could not reach health check endpoint: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /api/health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7a0f0ab7aba0>: Failed to establish a new connection: [Errno 111] Connection refused'))


NameError: name 'bn_voices_list' is not defined